In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import qmc

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

Using device: cuda
GPU: NVIDIA RTX 1000 Ada Generation Laptop GPU
VRAM: 6.44 GB

All imports successful


In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("GPU not detected, running on CPU")

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed(SEED)

In [2]:
def lorenz(t, state, sigma, rho, beta):
    """
    Lorenz system ODEs.
    
    Parameters
    ----------
    t     : float        - current time (required by solve_ivp signature)
    state : array (3,)   - [x, y, z]
    sigma : float        - Prandtl number
    rho   : float        - Rayleigh number
    beta  : float        - geometric factor
    
    Returns
    -------
    derivatives : list [dx/dt, dy/dt, dz/dt]
    """
    x, y, z = state
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return [dxdt, dydt, dzdt]


def solve_lorenz(sigma, rho, beta, t_span=(0, 10), n_steps=500, x0=[1.0, 1.0, 1.0]):
    """
    Integrate the Lorenz system for a given parameter set.
    
    Parameters
    ----------
    sigma, rho, beta : floats  - system parameters
    t_span           : tuple   - (t_start, t_end)
    n_steps          : int     - number of time points in output
    x0               : list    - initial condition [x0, y0, z0]
    
    Returns
    -------
    t : array (n_steps,)   - time points
    X : array (n_steps, 3) - states [x(t), y(t), z(t)]
    """
    t_eval = np.linspace(t_span[0], t_span[1], n_steps)
    
    sol = solve_ivp(
        fun=lorenz,
        t_span=t_span,
        y0=x0,
        t_eval=t_eval,
        args=(sigma, rho, beta),
        method="RK45",
        rtol=1e-8,
        atol=1e-8
    )
    
    return sol.t, sol.y.T  # sol.y is (3, n_steps), we transpose to (n_steps, 3)


# Quick sanity check with classic parameter values
sigma_ref = 10.0
rho_ref   = 28.0
beta_ref  = 8/3

t_check, X_check = solve_lorenz(sigma_ref, rho_ref, beta_ref)

print(f"Time array shape  : {t_check.shape}")
print(f"State array shape : {X_check.shape}")
print(f"\nFirst state  [x, y, z] : {X_check[0]}")
print(f"Last  state  [x, y, z] : {X_check[-1].round(4)}")

Time array shape  : (500,)
State array shape : (500, 3)

First state  [x, y, z] : [1. 1. 1.]
Last  state  [x, y, z] : [-4.9027 -3.7439 24.6908]


If you want inline plots in the notebook you can replace matplotlib.use("Agg") with %matplotlib inline at the top of the cell, but only do this if the kernel stays stable

In [6]:
import matplotlib
matplotlib.use("Agg") 

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Time series ---
ax1 = axes[0]
ax1.plot(t_check, X_check[:, 0], label="x(t)", lw=1.2)
ax1.plot(t_check, X_check[:, 1], label="y(t)", lw=1.2)
ax1.plot(t_check, X_check[:, 2], label="z(t)", lw=1.2)
ax1.set_title("State Variables over Time")
ax1.set_xlabel("t")
ax1.set_ylabel("state")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Phase portrait x vs z ---
ax2 = axes[1]
ax2.plot(X_check[:, 0], X_check[:, 2], lw=0.8, color="coral")
ax2.set_title("Phase Portrait  x vs z")
ax2.set_xlabel("x")
ax2.set_ylabel("z")
ax2.grid(True, alpha=0.3)

plt.suptitle(f"Lorenz System  |  sigma={sigma_ref}, rho={rho_ref}, beta={beta_ref:.3f}", 
             fontsize=13)
plt.tight_layout()
plt.savefig("lorenz_reference.png", dpi=150, bbox_inches="tight")
plt.show()

print("Plot saved to lorenz_reference.png")

Plot saved to lorenz_reference.png


C:\Users\muham\AppData\Local\Temp\ipykernel_6944\557366234.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
def sample_parameters(n_samples, seed=SEED):
    """
    Sample (sigma, rho, beta) using Latin Hypercube Sampling.
    
    Parameter ranges chosen to include both chaotic and non-chaotic regimes:
        sigma : [5,  15]
        rho   : [15, 35]
        beta  : [1,   4]
    
    Returns
    -------
    params : array (n_samples, 3)
    """
    sampler = qmc.LatinHypercube(d=3, seed=seed)
    unit_samples = sampler.random(n=n_samples)  # values in [0, 1]
    
    lower = [5.0, 15.0, 1.0]
    upper = [15.0, 35.0, 4.0]
    params = qmc.scale(unit_samples, lower, upper)
    
    return params


def generate_dataset(params, t_span=(0, 10), n_steps=500, x0=[1.0, 1.0, 1.0]):
    """
    For each parameter sample, solve the Lorenz system and collect
    (sigma, rho, beta, t) -> (x, y, z) pairs.
    
    Returns
    -------
    inputs  : array (n_samples * n_steps, 4)  - [sigma, rho, beta, t]
    targets : array (n_samples * n_steps, 3)  - [x, y, z]
    """
    inputs_list  = []
    targets_list = []
    
    for i, (sigma, rho, beta) in enumerate(params):
        t, X = solve_lorenz(sigma, rho, beta, t_span=t_span, n_steps=n_steps, x0=x0)
        
        # Build input rows: repeat (sigma, rho, beta) for each time step
        param_block = np.tile([sigma, rho, beta], (n_steps, 1))  # (n_steps, 3)
        t_col       = t.reshape(-1, 1)                            # (n_steps, 1)
        input_block = np.hstack([param_block, t_col])             # (n_steps, 4)
        
        inputs_list.append(input_block)
        targets_list.append(X)
        
        if (i + 1) % 50 == 0:
            print(f"  Solved {i + 1}/{len(params)} trajectories")
    
    inputs  = np.vstack(inputs_list)   # (n_samples * n_steps, 4)
    targets = np.vstack(targets_list)  # (n_samples * n_steps, 3)
    
    return inputs, targets

In [8]:
# --- Generate data ---
N_TRAIN = 200   # number of parameter samples for training
N_TEST  =  40   # held-out parameter samples never seen during training

print("Sampling parameter space...")
params_all   = sample_parameters(N_TRAIN + N_TEST)
params_train = params_all[:N_TRAIN]
params_test  = params_all[N_TRAIN:]

print(f"Train parameter samples : {len(params_train)}")
print(f"Test  parameter samples : {len(params_test)}")

print("\nGenerating training data...")
X_train_in, X_train_out = generate_dataset(params_train)

print("\nGenerating test data...")
X_test_in, X_test_out   = generate_dataset(params_test)

print(f"\nTraining set : inputs {X_train_in.shape},  targets {X_train_out.shape}")
print(f"Test set     : inputs {X_test_in.shape},  targets {X_test_out.shape}")
print(f"\nTotal training points : {len(X_train_in):,}")
print(f"Total test points     : {len(X_test_in):,}")

Sampling parameter space...
Train parameter samples : 200
Test  parameter samples : 40

Generating training data...
  Solved 50/200 trajectories
  Solved 100/200 trajectories
  Solved 150/200 trajectories
  Solved 200/200 trajectories

Generating test data...

Training set : inputs (100000, 4),  targets (100000, 3)
Test set     : inputs (20000, 4),  targets (20000, 3)

Total training points : 100,000
Total test points     : 20,000


In [9]:
class StandardScaler:
    """
    Simple z-score normalizer: transforms data to zero mean and unit variance.
    Fit on training data only, then applied to both train and test.
    """
    def __init__(self):
        self.mean = None
        self.std  = None

    def fit(self, data):
        self.mean = data.mean(axis=0)
        self.std  = data.std(axis=0)
        self.std[self.std < 1e-8] = 1.0  # avoid division by zero for constant features
        return self

    def transform(self, data):
        return (data - self.mean) / self.std

    def inverse_transform(self, data):
        return data * self.std + self.mean


class LorenzDataset(Dataset):
    """
    PyTorch Dataset wrapping normalized (inputs, targets) arrays.
    """
    def __init__(self, inputs, targets):
        # Store as float32 tensors on CPU, DataLoader will move batches to device
        self.inputs  = torch.tensor(inputs,  dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [10]:
# --- Fit scalers on training data only ---
scaler_in  = StandardScaler().fit(X_train_in)
scaler_out = StandardScaler().fit(X_train_out)

# --- Normalize ---
X_train_in_norm  = scaler_in.transform(X_train_in)
X_train_out_norm = scaler_out.transform(X_train_out)

X_test_in_norm   = scaler_in.transform(X_test_in)
X_test_out_norm  = scaler_out.transform(X_test_out)

# --- Build Datasets and DataLoaders ---
BATCH_SIZE = 1024

train_dataset = LorenzDataset(X_train_in_norm,  X_train_out_norm)
test_dataset  = LorenzDataset(X_test_in_norm,   X_test_out_norm)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# --- Sanity check ---
sample_in, sample_out = train_dataset[0]
print(f"Single sample input  shape : {sample_in.shape}   dtype: {sample_in.dtype}")
print(f"Single sample output shape : {sample_out.shape}  dtype: {sample_out.dtype}")

print(f"\nInput  means before normalization : {X_train_in.mean(axis=0).round(3)}")
print(f"Input  means after  normalization : {X_train_in_norm.mean(axis=0).round(3)}")
print(f"\nOutput means before normalization : {X_train_out.mean(axis=0).round(3)}")
print(f"Output means after  normalization : {X_train_out_norm.mean(axis=0).round(3)}")

print(f"\nTrain batches per epoch : {len(train_loader)}")
print(f"Test  batches           : {len(test_loader)}")

Single sample input  shape : torch.Size([4])   dtype: torch.float32
Single sample output shape : torch.Size([3])  dtype: torch.float32

Input  means before normalization : [10.053 25.106  2.525  5.   ]
Input  means after  normalization : [ 0. -0.  0.  0.]

Output means before normalization : [-4.126 -4.172 22.474]
Output means after  normalization : [ 0. -0. -0.]

Train batches per epoch : 98
Test  batches           : 20


In [11]:
class LorenzSurrogate(nn.Module):
    """
    Fully connected feedforward network mapping
    (sigma, rho, beta, t) -> (x, y, z).

    Parameters
    ----------
    input_dim   : int         - number of input features (4)
    output_dim  : int         - number of output features (3)
    hidden_dims : list[int]   - width of each hidden layer, length = depth
    activation  : nn.Module   - activation function applied after each hidden layer
    """
    def __init__(self, input_dim=4, output_dim=3,
                 hidden_dims=[64, 64, 64], activation=nn.Tanh()):
        super().__init__()

        self.activation = activation
        layer_sizes     = [input_dim] + hidden_dims + [output_dim]
        layers          = []

        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))
            # add activation after every layer except the last
            if i < len(layer_sizes) - 2:
                layers.append(self.activation)

        self.network = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for layer in self.network:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.network(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [12]:
# --- Build a reference model to inspect ---
model = LorenzSurrogate(
    input_dim   = 4,
    output_dim  = 3,
    hidden_dims = [128, 128, 128, 128],  # 4 hidden layers, 128 neurons each
    activation  = nn.Tanh()
).to(device)

print("Network architecture:")
print(model.network)
print(f"\nTotal trainable parameters : {count_parameters(model):,}")

# --- Verify forward pass with a random batch ---
dummy_input  = torch.randn(BATCH_SIZE, 4).to(device)
dummy_output = model(dummy_input)
print(f"\nDummy input  shape : {dummy_input.shape}")
print(f"Dummy output shape : {dummy_output.shape}")
print(f"Output sample      : {dummy_output[0].detach().cpu().numpy().round(4)}")

Network architecture:
Sequential(
  (0): Linear(in_features=4, out_features=128, bias=True)
  (1): Tanh()
  (2): Linear(in_features=128, out_features=128, bias=True)
  (3): Tanh()
  (4): Linear(in_features=128, out_features=128, bias=True)
  (5): Tanh()
  (6): Linear(in_features=128, out_features=128, bias=True)
  (7): Tanh()
  (8): Linear(in_features=128, out_features=3, bias=True)
)

Total trainable parameters : 50,563

Dummy input  shape : torch.Size([1024, 4])
Dummy output shape : torch.Size([1024, 3])
Output sample      : [-0.0865 -0.0131 -0.1392]


In [16]:
def train_model(model, train_loader, test_loader,
                lr=1e-3, epochs=100, device=device, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10
    )
    criterion = nn.MSELoss()
    history   = {"train_loss": [], "test_loss": []}

    for epoch in range(1, epochs + 1):

        # --- Training phase ---
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            predictions = model(x_batch)
            loss        = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(x_batch)

        train_loss /= len(train_loader.dataset)

        # --- Evaluation phase ---
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)
                predictions = model(x_batch)
                loss        = criterion(predictions, y_batch)
                test_loss  += loss.item() * len(x_batch)

        test_loss /= len(test_loader.dataset)

        scheduler.step(test_loss)
        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)

        if verbose and epoch % 10 == 0:
            print(f"Epoch {epoch:4d}/{epochs}  |  "
                  f"train loss: {train_loss:.6f}  |  "
                  f"test loss:  {test_loss:.6f}  |  "
                  f"lr: {optimizer.param_groups[0]['lr']:.2e}")

    return history

In [17]:
# --- Train the reference model ---
print("Training reference model: 4 hidden layers, 128 neurons, lr=1e-3, epochs=100")
print("-" * 70)

history = train_model(
    model        = model,
    train_loader = train_loader,
    test_loader  = test_loader,
    lr           = 1e-3,
    epochs       = 100,
    device       = device
)

print("-" * 70)
print(f"Final train loss : {history['train_loss'][-1]:.6f}")
print(f"Final test  loss : {history['test_loss'][-1]:.6f}")

Training reference model: 4 hidden layers, 128 neurons, lr=1e-3, epochs=100
----------------------------------------------------------------------
Epoch   10/100  |  train loss: 0.445938  |  test loss:  0.559025  |  lr: 1.00e-03
Epoch   20/100  |  train loss: 0.352782  |  test loss:  0.497035  |  lr: 1.00e-03
Epoch   30/100  |  train loss: 0.334104  |  test loss:  0.498308  |  lr: 5.00e-04
Epoch   40/100  |  train loss: 0.328186  |  test loss:  0.491446  |  lr: 5.00e-04
Epoch   50/100  |  train loss: 0.318132  |  test loss:  0.486559  |  lr: 5.00e-04
Epoch   60/100  |  train loss: 0.304908  |  test loss:  0.487754  |  lr: 2.50e-04
Epoch   70/100  |  train loss: 0.298373  |  test loss:  0.488140  |  lr: 1.25e-04
Epoch   80/100  |  train loss: 0.293812  |  test loss:  0.488806  |  lr: 1.25e-04
Epoch   90/100  |  train loss: 0.290722  |  test loss:  0.488688  |  lr: 6.25e-05
Epoch  100/100  |  train loss: 0.289186  |  test loss:  0.490381  |  lr: 3.13e-05
---------------------------------

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, len(history["train_loss"]) + 1)

# --- Loss curves ---
ax1 = axes[0]
ax1.plot(epochs_range, history["train_loss"], label="train loss", lw=1.5)
ax1.plot(epochs_range, history["test_loss"],  label="test loss",  lw=1.5, linestyle="--")
ax1.set_xlabel("epoch")
ax1.set_ylabel("MSE loss (normalized space)")
ax1.set_title("Training and Test Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Log scale loss (easier to see late-training behavior) ---
ax2 = axes[1]
ax2.semilogy(epochs_range, history["train_loss"], label="train loss", lw=1.5)
ax2.semilogy(epochs_range, history["test_loss"],  label="test loss",  lw=1.5, linestyle="--")
ax2.set_xlabel("epoch")
ax2.set_ylabel("MSE loss (log scale)")
ax2.set_title("Loss Curves (log scale)")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Mark where LR first decayed
ax1.axvline(x=30, color="gray", lw=0.8, linestyle=":", alpha=0.7)
ax2.axvline(x=30, color="gray", lw=0.8, linestyle=":", alpha=0.7)
ax1.text(31, 0.52, "LR decay", fontsize=8, color="gray")
ax2.text(31, 0.52, "LR decay", fontsize=8, color="gray")

plt.suptitle("Reference Model: 4 layers, 128 neurons, lr=1e-3", fontsize=12)
plt.tight_layout()
plt.savefig("loss_curves_reference.png", dpi=150, bbox_inches="tight")
plt.show()

print("Loss curves saved")

Loss curves saved


C:\Users\muham\AppData\Local\Temp\ipykernel_6944\978067736.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
def predict_trajectory(model, sigma, rho, beta, scaler_in, scaler_out,
                        t_span=(0, 10), n_steps=500, x0=[1.0, 1.0, 1.0]):
    """
    Run the DNN surrogate on a single parameter set and return
    predictions in the original (un-normalized) physical units.
    """
    t_eval = np.linspace(t_span[0], t_span[1], n_steps)

    # Build input array (n_steps, 4)
    param_block = np.tile([sigma, rho, beta], (n_steps, 1))
    t_col       = t_eval.reshape(-1, 1)
    inputs      = np.hstack([param_block, t_col])

    # Normalize
    inputs_norm = scaler_in.transform(inputs)

    # Forward pass
    model.eval()
    with torch.no_grad():
        inputs_tensor = torch.tensor(inputs_norm, dtype=torch.float32).to(device)
        preds_norm    = model(inputs_tensor).cpu().numpy()

    # Inverse transform back to physical units
    preds = scaler_out.inverse_transform(preds_norm)

    return t_eval, preds


# --- Pick a test parameter set (never seen during training) ---
sigma_t, rho_t, beta_t = params_test[0]
print(f"Test parameters: sigma={sigma_t:.3f}, rho={rho_t:.3f}, beta={beta_t:.3f}")

# --- Ground truth from solver ---
t_true, X_true = solve_lorenz(sigma_t, rho_t, beta_t)

# --- DNN prediction ---
t_pred, X_pred = predict_trajectory(model, sigma_t, rho_t, beta_t, scaler_in, scaler_out)

# --- Compute per-variable RMSE ---
rmse = np.sqrt(np.mean((X_true - X_pred) ** 2, axis=0))
print(f"\nRMSE  x: {rmse[0]:.4f}")
print(f"RMSE  y: {rmse[1]:.4f}")
print(f"RMSE  z: {rmse[2]:.4f}")
print(f"RMSE mean: {rmse.mean():.4f}")

# --- Plot ---
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
labels = ["x(t)", "y(t)", "z(t)"]

for i, ax in enumerate(axes):
    ax.plot(t_true, X_true[:, i], label="solver (truth)", lw=1.5)
    ax.plot(t_pred, X_pred[:, i], label="DNN prediction", lw=1.2,
            linestyle="--", alpha=0.85)
    ax.set_ylabel(labels[i])
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_title(f"{labels[i]}   RMSE={rmse[i]:.4f}")

axes[-1].set_xlabel("t")
plt.suptitle(
    f"Solver vs DNN  |  sigma={sigma_t:.2f}, rho={rho_t:.2f}, beta={beta_t:.2f}",
    fontsize=12
)
plt.tight_layout()
plt.savefig("prediction_vs_solver.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPrediction plot saved")

Test parameters: sigma=8.438, rho=34.437, beta=3.789

RMSE  x: 7.5249
RMSE  y: 9.6505
RMSE  z: 10.9005
RMSE mean: 9.3586

Prediction plot saved


C:\Users\muham\AppData\Local\Temp\ipykernel_6944\4158391054.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- x vs y ---
ax1 = axes[0]
ax1.plot(X_true[:, 0], X_true[:, 1], lw=0.7, label="solver", alpha=0.8)
ax1.plot(X_pred[:, 0], X_pred[:, 1], lw=0.7, label="DNN",    alpha=0.8, linestyle="--")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_title("Phase portrait  x vs y")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# --- x vs z ---
ax2 = axes[1]
ax2.plot(X_true[:, 0], X_true[:, 2], lw=0.7, label="solver", alpha=0.8)
ax2.plot(X_pred[:, 0], X_pred[:, 2], lw=0.7, label="DNN",    alpha=0.8, linestyle="--")
ax2.set_xlabel("x")
ax2.set_ylabel("z")
ax2.set_title("Phase portrait  x vs z")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# --- Pointwise error over time ---
ax3 = axes[2]
pointwise_error = np.sqrt(np.sum((X_true - X_pred) ** 2, axis=1))
ax3.plot(t_true, pointwise_error, lw=1.2, color="coral")
ax3.set_xlabel("t")
ax3.set_ylabel("||s_true - s_pred||")
ax3.set_title("Pointwise error over time")
ax3.grid(True, alpha=0.3)

# Mark short-time vs long-time boundary
t_mid = 5.0
ax3.axvline(x=t_mid, color="gray", lw=0.8, linestyle=":", alpha=0.7)
ax3.text(t_mid + 0.1, pointwise_error.max() * 0.9, "t=5", fontsize=8, color="gray")

# Short-time and long-time RMSE
t_idx = np.searchsorted(t_true, t_mid)
err_short = pointwise_error[:t_idx].mean()
err_long  = pointwise_error[t_idx:].mean()
print(f"Mean pointwise error  t < 5 : {err_short:.4f}")
print(f"Mean pointwise error  t > 5 : {err_long:.4f}")
print(f"Error growth factor         : {err_long / err_short:.2f}x")

plt.suptitle(
    f"Reference model  |  sigma={sigma_t:.2f}, rho={rho_t:.2f}, beta={beta_t:.2f}",
    fontsize=12
)
plt.tight_layout()
plt.savefig("phase_portrait_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Phase portrait comparison saved")

Mean pointwise error  t < 5 : 9.9207
Mean pointwise error  t > 5 : 19.0290
Error growth factor         : 1.92x
Phase portrait comparison saved


C:\Users\muham\AppData\Local\Temp\ipykernel_6944\3734350102.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# We will study three axes independently:
#   1. Network depth    : number of hidden layers
#   2. Learning rate    : initial lr passed to Adam
#   3. Epochs           : total training duration
#
# For each experiment we train a fresh model, record final test loss,
# and store the full history for plotting.

def run_experiment(hidden_dims, lr, epochs, label):
    """
    Build a fresh model, train it, evaluate on one test trajectory,
    and return everything needed for comparison.
    """
    print(f"\n--- {label} ---")
    print(f"    depth={len(hidden_dims)}, width={hidden_dims[0]}, lr={lr}, epochs={epochs}")

    m = LorenzSurrogate(
        input_dim   = 4,
        output_dim  = 3,
        hidden_dims = hidden_dims,
        activation  = nn.Tanh()
    ).to(device)

    hist = train_model(
        model        = m,
        train_loader = train_loader,
        test_loader  = test_loader,
        lr           = lr,
        epochs       = epochs,
        device       = device,
        verbose      = True
    )

    # Evaluate on the same test trajectory used earlier
    _, X_p = predict_trajectory(m, sigma_t, rho_t, beta_t, scaler_in, scaler_out)
    rmse   = float(np.sqrt(np.mean((X_true - X_p) ** 2)))

    print(f"    trajectory RMSE: {rmse:.4f}")

    return {"label": label, "history": hist, "model": m,
            "rmse": rmse, "X_pred": X_p,
            "hidden_dims": hidden_dims, "lr": lr, "epochs": epochs}


# --- Study 1: effect of depth (keep width=128, lr=1e-3, epochs=100) ---
depth_experiments = [
    run_experiment([128],                   lr=1e-3, epochs=100, label="depth=1"),
    run_experiment([128, 128],              lr=1e-3, epochs=100, label="depth=2"),
    run_experiment([128, 128, 128, 128],    lr=1e-3, epochs=100, label="depth=4  (reference)"),
    run_experiment([128]*6,                 lr=1e-3, epochs=100, label="depth=6"),
]


--- depth=1 ---
    depth=1, width=128, lr=0.001, epochs=100
Epoch   10/100  |  train loss: 0.659528  |  test loss:  0.758996  |  lr: 1.00e-03
Epoch   20/100  |  train loss: 0.626400  |  test loss:  0.719723  |  lr: 1.00e-03
Epoch   30/100  |  train loss: 0.610877  |  test loss:  0.707805  |  lr: 1.00e-03
Epoch   40/100  |  train loss: 0.596663  |  test loss:  0.693192  |  lr: 1.00e-03
Epoch   50/100  |  train loss: 0.586137  |  test loss:  0.683286  |  lr: 1.00e-03
Epoch   60/100  |  train loss: 0.578594  |  test loss:  0.681821  |  lr: 1.00e-03
Epoch   70/100  |  train loss: 0.573625  |  test loss:  0.672150  |  lr: 1.00e-03
Epoch   80/100  |  train loss: 0.568853  |  test loss:  0.673671  |  lr: 1.00e-03
Epoch   90/100  |  train loss: 0.565107  |  test loss:  0.670623  |  lr: 1.00e-03
Epoch  100/100  |  train loss: 0.561073  |  test loss:  0.674497  |  lr: 1.00e-03
    trajectory RMSE: 11.2726

--- depth=2 ---
    depth=2, width=128, lr=0.001, epochs=100
Epoch   10/100  |  train lo

In [22]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ["steelblue", "coral", "seagreen", "mediumpurple"]

# --- Test loss curves ---
ax1 = axes[0]
for exp, c in zip(depth_experiments, colors):
    ax1.plot(exp["history"]["test_loss"], label=exp["label"], lw=1.4, color=c)
ax1.set_xlabel("epoch")
ax1.set_ylabel("test loss (normalized MSE)")
ax1.set_title("Test Loss vs Epoch")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Train loss curves ---
ax2 = axes[1]
for exp, c in zip(depth_experiments, colors):
    ax2.plot(exp["history"]["train_loss"], label=exp["label"], lw=1.4,
             color=c, linestyle="--")
ax2.set_xlabel("epoch")
ax2.set_ylabel("train loss (normalized MSE)")
ax2.set_title("Train Loss vs Epoch")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# --- RMSE bar chart ---
ax3 = axes[2]
labels_d = [exp["label"] for exp in depth_experiments]
rmses_d  = [exp["rmse"]  for exp in depth_experiments]
bars = ax3.bar(labels_d, rmses_d, color=colors, alpha=0.85, width=0.5)
ax3.set_ylabel("trajectory RMSE (physical units)")
ax3.set_title("Trajectory RMSE by Depth")
ax3.grid(True, alpha=0.3, axis="y")

# Annotate bars
for bar, val in zip(bars, rmses_d):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.1,
             f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Depth Study  |  width=128, lr=1e-3, epochs=100", fontsize=12)
plt.tight_layout()
plt.savefig("depth_study.png", dpi=150, bbox_inches="tight")
plt.show()

print("Depth study plot saved")

Depth study plot saved


C:\Users\muham\AppData\Local\Temp\ipykernel_6944\3782400240.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
# --- Study 2: effect of learning rate ---
# Keep depth=4, width=128, epochs=100, vary lr

lr_experiments = [
    run_experiment([128]*4, lr=1e-2, epochs=100, label="lr=1e-2"),
    run_experiment([128]*4, lr=1e-3, epochs=100, label="lr=1e-3  (reference)"),
    run_experiment([128]*4, lr=1e-4, epochs=100, label="lr=1e-4"),
    run_experiment([128]*4, lr=1e-5, epochs=100, label="lr=1e-5"),
]


--- lr=1e-2 ---
    depth=4, width=128, lr=0.01, epochs=100
Epoch   10/100  |  train loss: 0.365885  |  test loss:  0.484847  |  lr: 1.00e-02
Epoch   20/100  |  train loss: 0.323870  |  test loss:  0.514466  |  lr: 1.00e-02
Epoch   30/100  |  train loss: 0.262772  |  test loss:  0.539079  |  lr: 5.00e-03
Epoch   40/100  |  train loss: 0.226873  |  test loss:  0.562803  |  lr: 2.50e-03
Epoch   50/100  |  train loss: 0.208984  |  test loss:  0.578644  |  lr: 1.25e-03
Epoch   60/100  |  train loss: 0.199205  |  test loss:  0.596906  |  lr: 6.25e-04
Epoch   70/100  |  train loss: 0.192939  |  test loss:  0.613208  |  lr: 3.13e-04
Epoch   80/100  |  train loss: 0.189685  |  test loss:  0.617886  |  lr: 1.56e-04
Epoch   90/100  |  train loss: 0.187880  |  test loss:  0.626093  |  lr: 7.81e-05
Epoch  100/100  |  train loss: 0.186867  |  test loss:  0.626955  |  lr: 3.91e-05
    trajectory RMSE: 8.6600

--- lr=1e-3  (reference) ---
    depth=4, width=128, lr=0.001, epochs=100
Epoch   10/100  

In [24]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ["coral", "seagreen", "steelblue", "mediumpurple"]

# --- Test loss curves ---
ax1 = axes[0]
for exp, c in zip(lr_experiments, colors):
    ax1.plot(exp["history"]["test_loss"], label=exp["label"], lw=1.4, color=c)
ax1.set_xlabel("epoch")
ax1.set_ylabel("test loss (normalized MSE)")
ax1.set_title("Test Loss vs Epoch")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Train vs test gap (generalization gap over time) ---
ax2 = axes[1]
for exp, c in zip(lr_experiments, colors):
    gap = np.array(exp["history"]["test_loss"]) - np.array(exp["history"]["train_loss"])
    ax2.plot(gap, label=exp["label"], lw=1.4, color=c)
ax2.set_xlabel("epoch")
ax2.set_ylabel("test loss - train loss")
ax2.set_title("Generalization Gap over Time")
ax2.axhline(y=0, color="gray", lw=0.8, linestyle=":")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# --- RMSE bar chart ---
ax3 = axes[2]
labels_lr = [exp["label"] for exp in lr_experiments]
rmses_lr  = [exp["rmse"]  for exp in lr_experiments]
bars = ax3.bar(labels_lr, rmses_lr, color=colors, alpha=0.85, width=0.5)
ax3.set_ylabel("trajectory RMSE (physical units)")
ax3.set_title("Trajectory RMSE by Learning Rate")
ax3.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, rmses_lr):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.1,
             f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Learning Rate Study  |  depth=4, width=128, epochs=100", fontsize=12)
plt.tight_layout()
plt.savefig("lr_study.png", dpi=150, bbox_inches="tight")
plt.show()

print("Learning rate study plot saved")

Learning rate study plot saved


C:\Users\muham\AppData\Local\Temp\ipykernel_6944\609550466.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
# --- Study 3: effect of epochs ---
# Keep depth=4, width=128, lr=1e-3, vary epochs
# This study directly connects to the earlier assignment observation

epoch_experiments = [
    run_experiment([128]*4, lr=1e-3, epochs=100,  label="epochs=100  (reference)"),
    run_experiment([128]*4, lr=1e-3, epochs=500,  label="epochs=500"),
    run_experiment([128]*4, lr=1e-3, epochs=1000, label="epochs=1000"),
]


--- epochs=100  (reference) ---
    depth=4, width=128, lr=0.001, epochs=100
Epoch   10/100  |  train loss: 0.433238  |  test loss:  0.541033  |  lr: 1.00e-03
Epoch   20/100  |  train loss: 0.353294  |  test loss:  0.484812  |  lr: 1.00e-03
Epoch   30/100  |  train loss: 0.341194  |  test loss:  0.486717  |  lr: 1.00e-03
Epoch   40/100  |  train loss: 0.321117  |  test loss:  0.482465  |  lr: 5.00e-04
Epoch   50/100  |  train loss: 0.310678  |  test loss:  0.500310  |  lr: 2.50e-04
Epoch   60/100  |  train loss: 0.304273  |  test loss:  0.495838  |  lr: 1.25e-04
Epoch   70/100  |  train loss: 0.300402  |  test loss:  0.497602  |  lr: 6.25e-05
Epoch   80/100  |  train loss: 0.298455  |  test loss:  0.497117  |  lr: 3.13e-05
Epoch   90/100  |  train loss: 0.297093  |  test loss:  0.495838  |  lr: 3.13e-05
Epoch  100/100  |  train loss: 0.296268  |  test loss:  0.497138  |  lr: 1.56e-05
    trajectory RMSE: 9.5165

--- epochs=500 ---
    depth=4, width=128, lr=0.001, epochs=500
Epoch   1

In [26]:
# Check where the model and data live during training
print(f"Device       : {device}")
print(f"Model device : {next(model.parameters()).device}")

# Confirm a batch actually moves to GPU during training
sample_batch_x, sample_batch_y = next(iter(train_loader))
sample_batch_x = sample_batch_x.to(device)
print(f"Batch device : {sample_batch_x.device}")

# GPU memory usage
if device.type == "cuda":
    allocated = torch.cuda.memory_allocated(0) / 1e6
    reserved  = torch.cuda.memory_reserved(0)  / 1e6
    print(f"\nGPU memory allocated : {allocated:.1f} MB")
    print(f"GPU memory reserved  : {reserved:.1f} MB")
    print(f"VRAM total           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Device       : cuda
Model device : cuda:0
Batch device : cuda:0

GPU memory allocated : 24.7 MB
GPU memory reserved  : 33.6 MB
VRAM total           : 6.44 GB


In [27]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ["steelblue", "coral", "seagreen"]

# --- Test loss curves (log scale to see the stall clearly) ---
ax1 = axes[0]
for exp, c in zip(epoch_experiments, colors):
    ax1.semilogy(exp["history"]["test_loss"], label=exp["label"], lw=1.4, color=c)
ax1.set_xlabel("epoch")
ax1.set_ylabel("test loss (log scale)")
ax1.set_title("Test Loss vs Epoch")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Learning rate over time ---
# Reconstruct LR schedule by re-running a dummy optimizer trace
ax2 = axes[1]
for exp, c in zip(epoch_experiments, colors):
    n_epochs = exp["epochs"]
    optimizer_trace = torch.optim.Adam(
        LorenzSurrogate().to(device).parameters(), lr=exp["lr"]
    )
    scheduler_trace = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_trace, mode="min", factor=0.5, patience=10
    )
    lr_trace = []
    for tl in exp["history"]["test_loss"]:
        lr_trace.append(optimizer_trace.param_groups[0]["lr"])
        scheduler_trace.step(tl)
    ax2.semilogy(lr_trace, label=exp["label"], lw=1.4, color=c)
ax2.set_xlabel("epoch")
ax2.set_ylabel("learning rate (log scale)")
ax2.set_title("Learning Rate Schedule")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# --- RMSE bar chart ---
ax3 = axes[2]
labels_e = [exp["label"] for exp in epoch_experiments]
rmses_e  = [exp["rmse"]  for exp in epoch_experiments]
bars = ax3.bar(labels_e, rmses_e, color=colors, alpha=0.85, width=0.5)
ax3.set_ylabel("trajectory RMSE (physical units)")
ax3.set_title("Trajectory RMSE by Epochs")
ax3.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, rmses_e):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.1,
             f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Epochs Study  |  depth=4, width=128, lr=1e-3", fontsize=12)
plt.tight_layout()
plt.savefig("epochs_study.png", dpi=150, bbox_inches="tight")
plt.show()

print("Epochs study plot saved")
print("\nDiagnosis: LR floor reached by epoch ~120 in both long runs")
print("Root cause: ReduceLROnPlateau with patience=10 decays too aggressively")
print("Fix: use CosineAnnealingLR instead, which keeps LR alive across all epochs")

Epochs study plot saved

Diagnosis: LR floor reached by epoch ~120 in both long runs
Root cause: ReduceLROnPlateau with patience=10 decays too aggressively
Fix: use CosineAnnealingLR instead, which keeps LR alive across all epochs


C:\Users\muham\AppData\Local\Temp\ipykernel_6944\1714158069.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
def train_model_fixed_lr(model, train_loader, test_loader,
                         lr=1e-3, epochs=3000, device=device,
                         verbose=True, print_every=100):
    """
    Training loop with fixed learning rate, no scheduler.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history   = {"train_loss": [], "test_loss": []}

    for epoch in range(1, epochs + 1):

        # --- Training phase ---
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            predictions = model(x_batch)
            loss        = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(x_batch)

        train_loss /= len(train_loader.dataset)

        # --- Evaluation phase ---
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)
                predictions = model(x_batch)
                loss        = criterion(predictions, y_batch)
                test_loss  += loss.item() * len(x_batch)

        test_loss /= len(test_loader.dataset)

        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)

        if verbose and epoch % print_every == 0:
            print(f"Epoch {epoch:5d}/{epochs}  |  "
                  f"train loss: {train_loss:.6f}  |  "
                  f"test loss: {test_loss:.6f}  |  "
                  f"lr: {lr:.2e}")

    return history


# Build final model
final_model = LorenzSurrogate(
    input_dim   = 4,
    output_dim  = 3,
    hidden_dims = [50] * 5,   # depth=5, width=50
    activation  = nn.Tanh()
).to(device)

print(f"Final model parameter count : {count_parameters(final_model):,}")
print(f"Reference model param count : {count_parameters(model):,}")
print()
print("Architecture:")
print(final_model.network)
print()
print("Training final model: depth=5, width=50, lr=1e-3, epochs=3000, no scheduler")
print("-" * 70)

history_final = train_model_fixed_lr(
    model        = final_model,
    train_loader = train_loader,
    test_loader  = test_loader,
    lr           = 1e-3,
    epochs       = 3000,
    device       = device,
    print_every  = 100
)

print("-" * 70)
print(f"Final train loss : {history_final['train_loss'][-1]:.6f}")
print(f"Final test  loss : {history_final['test_loss'][-1]:.6f}")

# --- Evaluate on test trajectory ---
_, X_pred_final = predict_trajectory(
    final_model, sigma_t, rho_t, beta_t, scaler_in, scaler_out
)
rmse_final = np.sqrt(np.mean((X_true - X_pred_final) ** 2))
print(f"Final trajectory RMSE : {rmse_final:.4f}")
print(f"Reference model RMSE  : 9.52  (depth=4, width=128, lr=1e-3, epochs=100)")

Final model parameter count : 10,603
Reference model param count : 50,563

Architecture:
Sequential(
  (0): Linear(in_features=4, out_features=50, bias=True)
  (1): Tanh()
  (2): Linear(in_features=50, out_features=50, bias=True)
  (3): Tanh()
  (4): Linear(in_features=50, out_features=50, bias=True)
  (5): Tanh()
  (6): Linear(in_features=50, out_features=50, bias=True)
  (7): Tanh()
  (8): Linear(in_features=50, out_features=50, bias=True)
  (9): Tanh()
  (10): Linear(in_features=50, out_features=3, bias=True)
)

Training final model: depth=5, width=50, lr=1e-3, epochs=3000, no scheduler
----------------------------------------------------------------------
